# NormLayer + AutoGen Demo

This notebook demonstrates how to use `AutoGenAdapter` to enforce behavioral policies
on an AutoGen-style agent's incoming and outgoing messages.

**No real AutoGen or LLM calls are needed** — we use mock objects to simulate the agent.

In [ ]:
import asyncio
from normlayer import PolicyEngine
from normlayer.policies.loop_detection import LoopDetection
from normlayer.policies.role_respect import RoleRespect
from normlayer.adapters.autogen_adapter import AutoGenAdapter

## 1. Define mock AutoGen objects

AutoGen uses async-only `on_messages()`. We simulate `TextMessage`, `Response`,
and an agent class.

In [ ]:
class MockTextMessage:
    def __init__(self, content, source):
        self.content = content
        self.source = source

class MockResponse:
    def __init__(self, chat_message, inner_messages=None):
        self.chat_message = chat_message
        self.inner_messages = inner_messages or []

class MockAutoGenAgent:
    def __init__(self, name, response):
        self.name = name
        self._response = response
    async def on_messages(self, messages, cancellation_token=None):
        return self._response

## 2. Set up the engine and adapter

In [ ]:
engine = PolicyEngine(
    policies=[
        LoopDetection(max_repetitions=2, similarity_threshold=0.9, handler="warn"),
        RoleRespect(
            role_definitions={"assistant": ["help", "answer", "explain"]},
            strict=True,
            handler="warn",
        ),
    ]
)

adapter = AutoGenAdapter(engine)
print("Engine policies:", [p.name for p in engine.policies])

## 3. Clean run — no violations

The adapter checks both incoming and outgoing messages.

In [ ]:
response = MockResponse(
    chat_message=MockTextMessage("I can help explain that concept.", "assistant")
)
agent = MockAutoGenAgent("assistant", response)
wrapped = adapter.wrap(agent)

incoming = [MockTextMessage("Can you explain transformers?", "user")]
result = await wrapped.on_messages(incoming)

print(f"Response: {result.chat_message.content}")
print(f"Violations: {len(engine.violations)}")

## 4. Trigger violations

Here the assistant agent produces output outside its defined role scope.

In [ ]:
engine2 = PolicyEngine(
    policies=[
        RoleRespect(
            role_definitions={"assistant": ["help", "answer"]},
            strict=True,
            handler="warn",
        ),
    ]
)
adapter2 = AutoGenAdapter(engine2)

bad_response = MockResponse(
    chat_message=MockTextMessage(
        "I will now execute the deployment and delete the database.", "assistant"
    )
)
bad_agent = MockAutoGenAgent("assistant", bad_response)
wrapped_bad = adapter2.wrap(bad_agent)

result2 = await wrapped_bad.on_messages(
    [MockTextMessage("Deploy the app", "user")]
)

print(f"\nViolations ({len(engine2.violations)}):")
for v in engine2.violations:
    print(f"  - {v.policy_violated} ({v.agent_id}): {v.message_snippet[:60]}")

## Summary

The `AutoGenAdapter` wraps an agent's async `on_messages()` method, checking both
incoming and outgoing messages. Unsupported message types (e.g., tool calls with
non-string content) are safely skipped.